# All Forces 2025 — TimesFM vs Prophet vs Baseline

Full year evaluation across all 43 territorial forces.  
We have complete 2025 actuals, so this is a clean held-out test.

**Training context:** 2012–2019 + 2022–2024  
**Test year:** 2025 (all 12 months, fully known)  
**Aggregation:** force level, all 16 crime types combined  
**Models:** TimesFM 2.5 (zero-shot) · Prophet (UK holidays) · Historical mean baseline

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import timesfm
from prophet import Prophet
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

## 1. Load data

In [ ]:
CRIME_TYPES = [
    'Violence and sexual offences', 'Criminal damage and arson', 'Drugs',
    'Violent crime', 'Burglary', 'Other theft', 'Vehicle crime',
    'Public order', 'Other crime', 'Shoplifting', 'Robbery',
    'Bicycle theft', 'Theft from the person', 'Possession of weapons',
    'Public disorder and weapons', 'Anti-social behaviour',
]
EXCLUDE_FORCES = {
    'British Transport Police',
    'Police Service of Northern Ireland',
}
TRAIN_YEARS = set(range(2012, 2020)) | set(range(2022, 2025))
TEST_MONTHS = pd.date_range('2025-01-01', periods=12, freq='MS')

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[
    raw['Crime type'].isin(CRIME_TYPES) &
    ~raw['Falls within'].isin(EXCLUDE_FORCES)
].copy()
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

force_monthly = (
    raw.groupby(['Falls within', 'month'])
    .size()
    .reset_index(name='count')
    .rename(columns={'Falls within': 'force'})
)

all_forces = sorted(force_monthly['force'].unique())
print(f'Forces: {len(all_forces)}')
print(f'Date range: {force_monthly["month"].min().date()} → {force_monthly["month"].max().date()}')
print(f'Test months: {TEST_MONTHS[0].date()} → {TEST_MONTHS[-1].date()}')

# Check all forces have full 2025 data
coverage = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['month'].nunique()
)
incomplete = coverage[coverage < 12]
if len(incomplete):
    print(f'\nWarning — forces with < 12 months of 2025 data:')
    print(incomplete)
else:
    print('\nAll forces have full 12 months of 2025 actuals.')

## 2. Load TimesFM

In [ ]:
print('Loading TimesFM...')
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(timesfm.ForecastConfig(
    max_context=512,
    max_horizon=128,
    per_core_batch_size=32,
    infer_is_positive=True,
    normalize_inputs=True,
))
print('Ready.')

## 3. TimesFM forecasts — all forces

In [ ]:
tfm_inputs = []
for force in all_forces:
    series = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .sort_values('month')['count']
        .values.astype(float)
    )
    tfm_inputs.append(series)

print('Running TimesFM batch forecast...')
tfm_point, _ = tfm.forecast(horizon=12, inputs=tfm_inputs)
tfm_arr = np.clip(np.array(tfm_point)[:, :12], 0, None)  # (43, 12)
print('Done.')

tfm_rows = []
for i, force in enumerate(all_forces):
    for j, month in enumerate(TEST_MONTHS):
        tfm_rows.append({'force': force, 'month': month, 'tfm_forecast': tfm_arr[i, j]})
tfm_df = pd.DataFrame(tfm_rows)
print(f'TimesFM forecast rows: {len(tfm_df):,}')

## 4. Prophet forecasts — all forces

In [ ]:
def run_prophet(train_series, n_months):
    df = train_series.reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = pd.to_datetime(df['ds'])
    m = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,
        seasonality_prior_scale=10,
        uncertainty_samples=0,
    )
    m.add_country_holidays(country_name='GB')
    m.fit(df)
    future = m.make_future_dataframe(periods=n_months, freq='MS', include_history=False)
    fc = m.predict(future)
    return np.clip(fc['yhat'].values[:n_months], 0, None)

prophet_rows = []
for i, force in enumerate(all_forces, 1):
    print(f'[{i:02d}/{len(all_forces)}] {force}', end='  ')
    train = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .set_index('month')['count']
        .sort_index()
    )
    preds = run_prophet(train, 12)
    print(f'done')
    for j, month in enumerate(TEST_MONTHS):
        prophet_rows.append({'force': force, 'month': month, 'prophet_forecast': preds[j]})

prophet_df = pd.DataFrame(prophet_rows)
print(f'Prophet forecast rows: {len(prophet_df):,}')

## 5. Assemble actuals, baseline, and all forecasts

In [ ]:
actuals = (
    force_monthly[force_monthly['month'].isin(TEST_MONTHS)]
    .rename(columns={'count': 'actual'})
)
baseline = (
    force_monthly[force_monthly['month'].dt.year.isin(TRAIN_YEARS)]
    .groupby('force')['count'].mean()
    .reset_index().rename(columns={'count': 'baseline'})
)

combined = (
    actuals
    .merge(baseline, on='force')
    .merge(tfm_df, on=['force', 'month'])
    .merge(prophet_df, on=['force', 'month'])
)
print(f'Combined rows: {len(combined):,}')
print(combined.head(3))

## 6. Evaluation metrics per force

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

rows = []
for force, g in combined.groupby('force'):
    a = g['actual'].values.astype(float)
    b = g['baseline'].values
    t = g['tfm_forecast'].values
    p = g['prophet_forecast'].values
    rows.append({
        'force':           force,
        'baseline_mae':    mae(b, a),  'baseline_rmse': rmse(b, a),
        'tfm_mae':         mae(t, a),  'tfm_rmse':      rmse(t, a),  'tfm_mape':     mape(t, a),
        'prophet_mae':     mae(p, a),  'prophet_rmse':  rmse(p, a),  'prophet_mape': mape(p, a),
        'mean_actual':     a.mean(),
    })

eval_df = pd.DataFrame(rows)
eval_df['tfm_rmae']     = eval_df['tfm_mae']     / eval_df['baseline_mae']
eval_df['prophet_rmae'] = eval_df['prophet_mae'] / eval_df['baseline_mae']
eval_df['tfm_wins']     = eval_df['tfm_mae']     < eval_df['baseline_mae']
eval_df['prophet_wins'] = eval_df['prophet_mae'] < eval_df['baseline_mae']
eval_df['better_model'] = np.where(
    eval_df['tfm_mae'] < eval_df['prophet_mae'], 'TimesFM', 'Prophet'
)
eval_df = eval_df.sort_values('tfm_mape').reset_index(drop=True)

# Headline summary
print('=== Overall Summary (2025, all 43 forces) ===')
print(f"Baseline  — MAE: {eval_df['baseline_mae'].mean():.1f}")
print(f"TimesFM   — MAE: {eval_df['tfm_mae'].mean():.1f}  MAPE: {eval_df['tfm_mape'].mean():.1f}%  "
      f"Win rate: {eval_df['tfm_wins'].mean()*100:.0f}%  RMAE: {eval_df['tfm_rmae'].mean():.3f}")
print(f"Prophet   — MAE: {eval_df['prophet_mae'].mean():.1f}  MAPE: {eval_df['prophet_mape'].mean():.1f}%  "
      f"Win rate: {eval_df['prophet_wins'].mean()*100:.0f}%  RMAE: {eval_df['prophet_rmae'].mean():.3f}")
print(f"\nTimesFM beats Prophet: {(eval_df['better_model']=='TimesFM').sum()}/{len(eval_df)} forces")
print(f"Prophet beats TimesFM: {(eval_df['better_model']=='Prophet').sum()}/{len(eval_df)} forces")
print()
display(
    eval_df[['force','mean_actual','baseline_mae','tfm_mae','tfm_mape','tfm_rmae',
             'prophet_mae','prophet_mape','prophet_rmae','better_model']]
    .round(2)
    .rename(columns={
        'force':'Force','mean_actual':'Avg Monthly Actual',
        'baseline_mae':'Baseline MAE',
        'tfm_mae':'TFM MAE','tfm_mape':'TFM MAPE%','tfm_rmae':'TFM RMAE',
        'prophet_mae':'Prophet MAE','prophet_mape':'Prophet MAPE%','prophet_rmae':'Prophet RMAE',
        'better_model':'Winner',
    })
    .set_index('Force')
)

## 7. MAPE comparison chart — all forces

In [ ]:
sorted_df = eval_df.sort_values('tfm_mape')
force_labels = (
    sorted_df['force']
    .str.replace(' Constabulary', '').str.replace(' Police', '').str.replace(' Service', '')
)

fig, ax = plt.subplots(figsize=(11, 14))
y = np.arange(len(sorted_df))
w = 0.35
ax.barh(y - w/2, sorted_df['tfm_mape'],     w, label='TimesFM', color='#2196F3', alpha=0.85)
ax.barh(y + w/2, sorted_df['prophet_mape'], w, label='Prophet',  color='#4CAF50', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(force_labels, fontsize=8)
ax.axvline(5,  color='#FFC107', linestyle='--', linewidth=1, label='5%')
ax.axvline(10, color='orange',  linestyle='--', linewidth=1, label='10%')
ax.axvline(20, color='red',     linestyle='--', linewidth=1, label='20%')
ax.set_xlabel('MAPE % (lower = better)')
ax.set_title('2025 Evaluation — TimesFM vs Prophet MAPE % by Force\n(full year, all 43 forces)', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/2025_mape_comparison_all_forces.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. MAE vs Baseline — grouped bar chart

In [ ]:
sorted_mae = eval_df.sort_values('baseline_mae', ascending=False)
fl = (
    sorted_mae['force']
    .str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service','')
)

fig, ax = plt.subplots(figsize=(11, 14))
y = np.arange(len(sorted_mae))
w = 0.27
ax.barh(y + w,   sorted_mae['baseline_mae'],    w, label='Baseline', color='#9E9E9E', alpha=0.9)
ax.barh(y,       sorted_mae['tfm_mae'],         w, label='TimesFM',  color='#2196F3', alpha=0.9)
ax.barh(y - w,   sorted_mae['prophet_mae'],     w, label='Prophet',   color='#4CAF50', alpha=0.9)
ax.set_yticks(y)
ax.set_yticklabels(fl, fontsize=8)
ax.set_xlabel('MAE (lower = better)')
ax.set_title('2025 Evaluation — MAE by Force\nBaseline vs TimesFM vs Prophet', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/2025_mae_comparison_all_forces.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. RMAE heatmap — which model is better where

In [ ]:
rmae_data = eval_df[['force','tfm_rmae','prophet_rmae']].set_index('force')
rmae_data.index = (
    rmae_data.index
    .str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service','')
)
rmae_data = rmae_data.sort_values('tfm_rmae')
rmae_data.columns = ['TimesFM RMAE', 'Prophet RMAE']

fig, ax = plt.subplots(figsize=(5, 14))
sns.heatmap(
    rmae_data.round(2), annot=True, fmt='.2f', cmap='RdYlGn_r',
    center=1.0, vmin=0.5, vmax=1.5,
    linewidths=0.4, ax=ax,
    cbar_kws={'label': 'RMAE (< 1.0 = beats baseline)'}
)
ax.set_title('RMAE by Force — 2025\n(green < 1.0 = beats baseline)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/2025_rmae_heatmap_all_forces.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Forecast vs Actual grid — all forces (full year 2025)

In [ ]:
HISTORY_START = '2022-01-01'
plot_forces = sorted(combined['force'].unique())  # only forces present in combined
ncols = 5
nrows = int(np.ceil(len(plot_forces) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(22, nrows * 3.8))
axes_flat = axes.flatten()
fig.suptitle('2025 Full Year — Actual vs TimesFM vs Prophet vs Baseline (All Forces)',
             fontsize=13, fontweight='bold')

for i, force in enumerate(plot_forces):
    ax = axes_flat[i]

    # Recent history
    history = force_monthly[
        (force_monthly['force'] == force) &
        (force_monthly['month'] >= HISTORY_START) &
        (force_monthly['month'].dt.year < 2025)
    ].sort_values('month')
    ax.plot(history['month'], history['count'], color='#94a3b8',
            linewidth=1, label='History', zorder=1)

    fc = combined[combined['force'] == force].sort_values('month')
    if fc.empty:
        ax.set_visible(False)
        continue

    # Baseline (flat line)
    bl_val = fc['baseline'].iloc[0]
    ax.axhline(bl_val, color='#9E9E9E', linestyle=':', linewidth=1, label='Baseline', zorder=2)

    # TimesFM
    ax.plot(fc['month'], fc['tfm_forecast'], color='#2196F3', linewidth=1.8,
            linestyle='--', marker='o', markersize=3, label='TimesFM', zorder=4)

    # Prophet
    ax.plot(fc['month'], fc['prophet_forecast'], color='#4CAF50', linewidth=1.8,
            linestyle='--', marker='s', markersize=3, label='Prophet', zorder=4)

    # Actuals
    ax.plot(fc['month'], fc['actual'], color='#FF5722', linewidth=2.2,
            marker='o', markersize=4, label='Actual', zorder=5)

    ax.set_title(
        force.replace(' Constabulary','').replace(' Police','').replace(' Service',''),
        fontsize=8, fontweight='bold'
    )
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.tick_params(axis='x', labelsize=6, rotation=30)
    ax.tick_params(axis='y', labelsize=6)

    # Annotate MAPE for each model
    row = eval_df[eval_df['force'] == force]
    if not row.empty:
        tfm_m  = row['tfm_mape'].values[0]
        prop_m = row['prophet_mape'].values[0]
        ax.text(0.02, 0.97,
                f'TFM {tfm_m:.1f}%\nPro {prop_m:.1f}%',
                transform=ax.transAxes, ha='left', va='top', fontsize=6,
                color='#0d47a1',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig('../outputs/2025_forecast_vs_actual_all_forces.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Monthly aggregate — national totals

In [ ]:
national = combined.groupby('month').agg(
    actual=('actual','sum'),
    tfm_forecast=('tfm_forecast','sum'),
    prophet_forecast=('prophet_forecast','sum'),
    baseline=('baseline','sum'),
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(national['month'], national['actual'],          color='#FF5722', linewidth=2.5,
        marker='o', markersize=5, label='Actual', zorder=5)
ax.plot(national['month'], national['tfm_forecast'],    color='#2196F3', linewidth=2,
        linestyle='--', marker='o', markersize=4, label='TimesFM', zorder=4)
ax.plot(national['month'], national['prophet_forecast'],color='#4CAF50', linewidth=2,
        linestyle='--', marker='s', markersize=4, label='Prophet', zorder=4)
ax.axhline(national['baseline'].mean(), color='#9E9E9E', linestyle=':',
           linewidth=1.5, label='Baseline')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.tick_params(axis='x', rotation=30)
ax.set_title('2025 National Monthly Totals — All 43 Forces Combined', fontweight='bold', fontsize=12)
ax.set_ylabel('Total incidents')
ax.legend()

# Annotate national MAPE
nat_tfm_mape  = mape(national['tfm_forecast'].values, national['actual'].values)
nat_prop_mape = mape(national['prophet_forecast'].values, national['actual'].values)
ax.text(0.02, 0.97, f'National MAPE — TimesFM: {nat_tfm_mape:.1f}%  Prophet: {nat_prop_mape:.1f}%',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#cbd5e1'))

plt.tight_layout()
plt.savefig('../outputs/2025_national_monthly_totals.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Winner map — which model wins per force

In [ ]:
winner_counts = eval_df['better_model'].value_counts()
print('Model wins (lower MAE):')
print(winner_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('2025 Model Comparison Summary — All 43 Forces', fontsize=12, fontweight='bold')

# Pie: overall winner count
axes[0].pie(
    winner_counts.values, labels=winner_counts.index,
    colors=['#2196F3', '#4CAF50'], autopct='%1.0f%%',
    startangle=90, textprops={'fontsize': 12}
)
axes[0].set_title('Which model wins more forces?', fontsize=10)

# Scatter: TFM MAPE vs Prophet MAPE per force
axes[1].scatter(
    eval_df['tfm_mape'], eval_df['prophet_mape'],
    c=['#2196F3' if b == 'TimesFM' else '#4CAF50' for b in eval_df['better_model']],
    s=60, alpha=0.8, edgecolors='white', linewidth=0.5
)
# Diagonal: y=x line (equal performance)
lim = max(eval_df['tfm_mape'].max(), eval_df['prophet_mape'].max()) * 1.05
axes[1].plot([0, lim], [0, lim], color='#9E9E9E', linestyle='--', linewidth=1)
axes[1].set_xlabel('TimesFM MAPE %')
axes[1].set_ylabel('Prophet MAPE %')
axes[1].set_title('TFM vs Prophet MAPE per Force\n(below diagonal = TimesFM better)', fontsize=10)

# Label outliers
for _, r in eval_df.iterrows():
    if r['tfm_mape'] > 15 or r['prophet_mape'] > 15:
        label = r['force'].replace(' Constabulary','').replace(' Police','').replace(' Service','')
        axes[1].annotate(label, (r['tfm_mape'], r['prophet_mape']),
                         fontsize=6, ha='left', va='bottom')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2196F3', label='TimesFM wins'),
                   Patch(facecolor='#4CAF50', label='Prophet wins')]
axes[1].legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/2025_model_winner_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save outputs

In [ ]:
os.makedirs('../outputs', exist_ok=True)
combined.to_parquet('../outputs/2025_all_forces_actuals_vs_forecasts.parquet', index=False)
eval_df.to_csv('../outputs/2025_all_forces_evaluation.csv', index=False)
print('Saved:')
print('  2025_all_forces_actuals_vs_forecasts.parquet — monthly actual/tfm/prophet/baseline per force')
print('  2025_all_forces_evaluation.csv               — full evaluation metrics per force')